# 10 - Silver: Annual Bilateral Trade Partner Fact Table

Creates `silver.fact_trade_partner_annual` — one row per reporter × counterpart × year.

## Source priority

| Reporter × Year | Condition | Source |
|----------------|-----------|--------|
| Any year | `comtrade_country_year_coverage.quality_flag = 'good'` | UN Comtrade (`un_comtrade`) |
| Fallback | Comtrade missing / sparse / dark / anomaly | IMF DOTS (`imf_imts`) |

The coverage quality flag (from notebook 10b's `silver.comtrade_country_year_coverage`) drives
which source is used for each reporter×year. This is stricter than a simple year cutoff.

## Source attribution columns

| Column | Values | Meaning |
|--------|--------|---------|
| `trade_source` | `imf_imts` / `un_comtrade` | Source system |
| `selected_source` | `IMTS_DOTS` / `UN_COMTRADE_DIRECT` | Human-readable label |
| `is_comtrade_backed` | bool | Row came from HS6-aggregated Comtrade data |
| `is_fallback` | bool | Comtrade quality_flag was not 'good'; DOTS used instead |
| `coverage_method` | string | How the value was derived |

## Inputs

- `bronze.imts_raw` — IMF DOTS bilateral trade
- `silver.comtrade_partner_annual` ← produced by notebook 10b
- `silver.comtrade_country_year_coverage` ← produced by notebook 10b
- `silver.dim_country`
- `silver.dim_bloc_membership`


In [ ]:
# Cell 1 - Configuration
from datetime import datetime, timezone

from pyspark.sql import Window
from pyspark.sql import functions as F

spark.sql("USE CATALOG cemac_ecowas_aes_trade")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

START_YEAR = 1990
END_YEAR = 2024
EXPORT_INDICATOR = "XG_FOB_USD"
IMPORT_INDICATOR = "MG_CIF_USD"
WORLD_COUNTERPART_CODE = "W00"

# Years where Comtrade bilateral data takes priority over DOTS for reporters
# that have real bilateral coverage (determined in notebook 10b).
COMTRADE_PRIORITY_FROM = 2022

loaded_at = datetime.now(timezone.utc)

print("Target table  : silver.fact_trade_partner_annual")
print(f"Years         : {START_YEAR}–{END_YEAR}")
print(f"DOTS priority : {START_YEAR}–{COMTRADE_PRIORITY_FROM - 1}")
print(f"Comtrade priority: {COMTRADE_PRIORITY_FROM}–{END_YEAR} (where coverage exists)")

In [ ]:
# Cell 2 - Input table checks
def table_exists(schema_name, table_name):
    return spark.sql(f"SHOW TABLES IN {schema_name} LIKE '{table_name}'").count() > 0


required_tables = [
    ("bronze", "imts_raw"),
    ("silver", "comtrade_partner_annual"),
    ("silver", "comtrade_country_year_coverage"),
    ("silver", "dim_country"),
    ("silver", "dim_bloc_membership"),
]

missing_tables = [f"{schema}.{table}" for schema, table in required_tables if not table_exists(schema, table)]
if missing_tables:
    raise ValueError(f"Missing required input table(s): {missing_tables}")

print("All required tables found.")

print("\nDOTS input coverage:")
spark.sql("""
    SELECT
        indicator,
        COUNT(*) AS rows,
        COUNT(DISTINCT reporter_iso3) AS reporters,
        MIN(year) AS earliest_year,
        MAX(year) AS latest_year
    FROM bronze.imts_raw
    GROUP BY indicator
    ORDER BY indicator
""").show(truncate=False)

print("Comtrade coverage by quality_flag:")
spark.sql("""
    SELECT quality_flag, COUNT(*) AS country_years, COUNT(DISTINCT country_iso3) AS countries
    FROM silver.comtrade_country_year_coverage
    GROUP BY quality_flag
    ORDER BY country_years DESC
""").show(truncate=False)

print("Reporter×years where Comtrade quality_flag = 'good':")
spark.sql("""
    SELECT country_iso3, MIN(year) AS first_good_year, MAX(year) AS last_good_year, COUNT(*) AS good_years
    FROM silver.comtrade_country_year_coverage
    WHERE quality_flag = 'good'
    GROUP BY country_iso3
    ORDER BY country_iso3
""").show(truncate=False)


In [ ]:
# Cell 3 - Normalize bronze IMTS rows and pivot exports/imports
raw_df = (
    spark.table("bronze.imts_raw")
    .select(
        F.upper(F.trim(F.col("reporter_iso3"))).alias("reporter_iso3"),
        F.upper(F.trim(F.col("counterpart_iso3"))).alias("counterpart_iso3"),
        F.upper(F.trim(F.col("indicator"))).alias("indicator"),
        F.col("year").cast("int").alias("year"),
        F.col("value_usd").cast("double").alias("value_usd"),
    )
    .where((F.col("year") >= START_YEAR) & (F.col("year") <= END_YEAR))
    .where(F.col("indicator").isin([EXPORT_INDICATOR, IMPORT_INDICATOR]))
    .where(F.col("reporter_iso3").isNotNull() & (F.col("reporter_iso3") != ""))
    .where(F.col("counterpart_iso3").isNotNull() & (F.col("counterpart_iso3") != ""))
)

# Sum is deliberate: if the IMF API returns duplicate slices in a future release, silver remains stable.
pivot_df = (
    raw_df
    .groupBy("reporter_iso3", "counterpart_iso3", "year")
    .agg(
        F.sum(F.when(F.col("indicator") == EXPORT_INDICATOR, F.col("value_usd"))).alias("exports_fob_usd"),
        F.sum(F.when(F.col("indicator") == IMPORT_INDICATOR, F.col("value_usd"))).alias("imports_cif_usd"),
    )
    .withColumn("is_world_aggregate", F.col("counterpart_iso3") == F.lit(WORLD_COUNTERPART_CODE))
    .withColumn("total_trade_usd", F.coalesce(F.col("exports_fob_usd"), F.lit(0.0)) + F.coalesce(F.col("imports_cif_usd"), F.lit(0.0)))
    .withColumn("trade_balance_usd", F.coalesce(F.col("exports_fob_usd"), F.lit(0.0)) - F.coalesce(F.col("imports_cif_usd"), F.lit(0.0)))
    .withColumn("exports_billions_usd", F.col("exports_fob_usd") / F.lit(1_000_000_000.0))
    .withColumn("imports_billions_usd", F.col("imports_cif_usd") / F.lit(1_000_000_000.0))
    .withColumn("total_trade_billions_usd", F.col("total_trade_usd") / F.lit(1_000_000_000.0))
    .withColumn("trade_balance_billions_usd", F.col("trade_balance_usd") / F.lit(1_000_000_000.0))
)

print(f"Reporter-counterpart-year rows: {pivot_df.count():,}")
pivot_df.show(10, truncate=False)

In [ ]:
# Cell 4 - Join reporter and project-counterpart dimensions
country_dim = spark.table("silver.dim_country")

reporter_dim = country_dim.select(
    F.col("country_key").alias("reporter_country_key"),
    F.col("country_iso3").alias("reporter_iso3_dim"),
    F.col("country_name").alias("reporter_name"),
    F.col("region").alias("reporter_region"),
)

counterpart_dim = country_dim.select(
    F.col("country_key").alias("counterpart_country_key"),
    F.col("country_iso3").alias("counterpart_iso3_dim"),
    F.col("country_name").alias("counterpart_project_country_name"),
    F.col("region").alias("counterpart_project_region"),
)

reporter_bloc = spark.table("silver.dim_bloc_membership").where("is_primary_analytical_bloc = true").select(
    F.col("country_iso3").alias("reporter_bloc_iso3"),
    F.col("bloc_code").alias("reporter_bloc_code"),
    F.col("bloc_name").alias("reporter_bloc_name"),
    F.col("valid_from").alias("reporter_bloc_valid_from"),
    F.col("valid_to").alias("reporter_bloc_valid_to"),
)

counterpart_bloc = spark.table("silver.dim_bloc_membership").where("is_primary_analytical_bloc = true").select(
    F.col("country_iso3").alias("counterpart_bloc_iso3"),
    F.col("bloc_code").alias("counterpart_bloc_code"),
    F.col("bloc_name").alias("counterpart_bloc_name"),
    F.col("valid_from").alias("counterpart_bloc_valid_from"),
    F.col("valid_to").alias("counterpart_bloc_valid_to"),
)

enriched_df = (
    pivot_df.withColumn("year_end_date", F.expr("make_date(year, 12, 31)"))
    .join(reporter_dim, F.col("reporter_iso3") == F.col("reporter_iso3_dim"), "left")
    .join(counterpart_dim, F.col("counterpart_iso3") == F.col("counterpart_iso3_dim"), "left")
    .join(
        reporter_bloc,
        (F.col("reporter_iso3") == F.col("reporter_bloc_iso3"))
        & (F.col("year_end_date") >= F.col("reporter_bloc_valid_from"))
        & (F.col("reporter_bloc_valid_to").isNull() | (F.col("year_end_date") <= F.col("reporter_bloc_valid_to"))),
        "left",
    )
    .join(
        counterpart_bloc,
        (F.col("counterpart_iso3") == F.col("counterpart_bloc_iso3"))
        & (F.col("year_end_date") >= F.col("counterpart_bloc_valid_from"))
        & (F.col("counterpart_bloc_valid_to").isNull() | (F.col("year_end_date") <= F.col("counterpart_bloc_valid_to"))),
        "left",
    )
    .withColumn("counterpart_is_project_country", F.col("counterpart_country_key").isNotNull())
    .withColumn(
        "counterpart_name",
        F.when(F.col("is_world_aggregate"), F.lit("World"))
        .when(F.col("counterpart_project_country_name").isNotNull(), F.col("counterpart_project_country_name"))
        .otherwise(F.col("counterpart_iso3")),
    )
    .withColumn(
        "counterpart_group_code",
        F.when(F.col("is_world_aggregate"), F.lit("WORLD"))
        .when(F.col("counterpart_bloc_code").isNotNull(), F.col("counterpart_bloc_code"))
        .otherwise(F.lit("REST_OF_WORLD")),
    )
    .withColumn(
        "counterpart_group_name",
        F.when(F.col("is_world_aggregate"), F.lit("World aggregate"))
        .when(F.col("counterpart_bloc_name").isNotNull(), F.col("counterpart_bloc_name"))
        .otherwise(F.lit("Rest of world")),
    )
)

missing_reporters = enriched_df.where(F.col("reporter_country_key").isNull()).select("reporter_iso3").distinct().count()
if missing_reporters:
    enriched_df.where(F.col("reporter_country_key").isNull()).select("reporter_iso3").distinct().show(truncate=False)
    raise ValueError(f"Reporter countries missing from silver.dim_country: {missing_reporters}")

print("Dimension joins complete.")
enriched_df.select("reporter_iso3", "reporter_name", "counterpart_iso3", "counterpart_name", "counterpart_group_code", "year").show(20, truncate=False)

In [ ]:
# Cell 5 - Coverage-aware merge: DOTS baseline + Comtrade (quality_flag = 'good')
#
# Strategy:
#   A. Build DOTS baseline with partner-share metrics; tag as imf_imts / IMTS_DOTS.
#   B. Load Comtrade partner annual (long format: one row per flow_type).
#      Join coverage table → filter to quality_flag = 'good' only.
#      Pivot export/import flows to wide schema matching DOTS.
#   C. Enrich Comtrade rows with reporter/counterpart dimensions + partner-share metrics.
#   D. Identify reporter×year cells where Comtrade is valid.
#      Drop DOTS rows for those cells; substitute Comtrade rows.
#      Tag all rows with source attribution columns.

# ─── Step A: DOTS baseline ────────────────────────────────────────────────────
dots_base = (
    enriched_df
    .withColumn(
        "reporter_year_exports_partner_sum_usd",
        F.sum(
            F.when(~F.col("is_world_aggregate"), F.coalesce(F.col("exports_fob_usd"), F.lit(0.0))).otherwise(F.lit(0.0))
        ).over(Window.partitionBy("reporter_iso3", "year")),
    )
    .withColumn(
        "reporter_year_imports_partner_sum_usd",
        F.sum(
            F.when(~F.col("is_world_aggregate"), F.coalesce(F.col("imports_cif_usd"), F.lit(0.0))).otherwise(F.lit(0.0))
        ).over(Window.partitionBy("reporter_iso3", "year")),
    )
    .withColumn(
        "reporter_year_total_trade_partner_sum_usd",
        F.sum(
            F.when(~F.col("is_world_aggregate"), F.coalesce(F.col("total_trade_usd"), F.lit(0.0))).otherwise(F.lit(0.0))
        ).over(Window.partitionBy("reporter_iso3", "year")),
    )
    .withColumn(
        "export_partner_share_pct",
        F.when(
            (~F.col("is_world_aggregate")) & (F.col("reporter_year_exports_partner_sum_usd") > 0),
            F.col("exports_fob_usd") / F.col("reporter_year_exports_partner_sum_usd") * 100.0,
        ),
    )
    .withColumn(
        "import_partner_share_pct",
        F.when(
            (~F.col("is_world_aggregate")) & (F.col("reporter_year_imports_partner_sum_usd") > 0),
            F.col("imports_cif_usd") / F.col("reporter_year_imports_partner_sum_usd") * 100.0,
        ),
    )
    .withColumn(
        "total_trade_partner_share_pct",
        F.when(
            (~F.col("is_world_aggregate")) & (F.col("reporter_year_total_trade_partner_sum_usd") > 0),
            F.col("total_trade_usd") / F.col("reporter_year_total_trade_partner_sum_usd") * 100.0,
        ),
    )
    .withColumn("trade_source", F.lit("imf_imts"))
    .withColumn("selected_source", F.lit("IMTS_DOTS"))
    .withColumn("is_comtrade_backed", F.lit(False))
    .withColumn("is_fallback", F.lit(False))
    .withColumn("coverage_method", F.lit("imts_partner_annual"))
    .withColumn("hs6_code_count", F.lit(None).cast("long"))
    .withColumn("loaded_at", F.lit(loaded_at))
)

# ─── Step B: Load Comtrade partner annual + coverage, filter to quality_flag = 'good' ─
coverage_tbl = spark.table("silver.comtrade_country_year_coverage").select(
    "country_iso3", "year", "quality_flag", "recommended_action",
)

comtrade_long = spark.table("silver.comtrade_partner_annual")

# Filter to rows where the reporter×year coverage is 'good'
comtrade_good_long = (
    comtrade_long
    .join(
        coverage_tbl,
        (comtrade_long.reporter_iso3 == coverage_tbl.country_iso3)
        & (comtrade_long.year == coverage_tbl.year),
        "left",
    )
    .drop("country_iso3")
    .filter(F.col("quality_flag") == "good")
)

# Pivot long (export/import rows) → wide (exports_fob_usd / imports_cif_usd columns)
comtrade_wide = (
    comtrade_good_long
    .groupBy("reporter_iso3", "partner_iso3", "year", "quality_flag")
    .agg(
        F.sum(F.when(F.col("flow_type") == "export", F.col("trade_value_usd"))).alias("exports_fob_usd"),
        F.sum(F.when(F.col("flow_type") == "import", F.col("trade_value_usd"))).alias("imports_cif_usd"),
        F.sum(F.col("distinct_hs6_products")).alias("hs6_code_count"),
    )
    .withColumn("is_world_aggregate", F.lit(False))
    .withColumn(
        "total_trade_usd",
        F.coalesce(F.col("exports_fob_usd"), F.lit(0.0)) + F.coalesce(F.col("imports_cif_usd"), F.lit(0.0)),
    )
    .withColumn(
        "trade_balance_usd",
        F.coalesce(F.col("exports_fob_usd"), F.lit(0.0)) - F.coalesce(F.col("imports_cif_usd"), F.lit(0.0)),
    )
    .withColumn("exports_billions_usd", F.col("exports_fob_usd") / F.lit(1_000_000_000.0))
    .withColumn("imports_billions_usd", F.col("imports_cif_usd") / F.lit(1_000_000_000.0))
    .withColumn("total_trade_billions_usd", F.col("total_trade_usd") / F.lit(1_000_000_000.0))
    .withColumn("trade_balance_billions_usd", F.col("trade_balance_usd") / F.lit(1_000_000_000.0))
)

# ─── Step C: Enrich Comtrade rows with reporter/counterpart dimensions ────────
country_dim_c = spark.table("silver.dim_country")

comtrade_enriched = (
    comtrade_wide
    .join(
        country_dim_c.select(
            F.col("country_key").alias("counterpart_country_key"),
            F.col("country_iso3").alias("ct_counterpart_iso3"),
            F.col("country_name").alias("counterpart_project_country_name"),
            F.col("region").alias("counterpart_project_region"),
        ),
        F.col("partner_iso3") == F.col("ct_counterpart_iso3"),
        "left",
    )
    .join(
        country_dim_c.select(
            F.col("country_key").alias("reporter_country_key"),
            F.col("country_iso3").alias("ct_reporter_iso3"),
            F.col("country_name").alias("reporter_name"),
            F.col("region").alias("reporter_region"),
        ),
        F.col("reporter_iso3") == F.col("ct_reporter_iso3"),
        "left",
    )
    .join(
        spark.table("silver.dim_bloc_membership").where("is_primary_analytical_bloc = true").select(
            F.col("country_iso3").alias("reporter_bloc_iso3"),
            F.col("bloc_code").alias("reporter_bloc_code"),
            F.col("bloc_name").alias("reporter_bloc_name"),
            F.col("valid_from").alias("reporter_bloc_valid_from"),
            F.col("valid_to").alias("reporter_bloc_valid_to"),
        ),
        (F.col("reporter_iso3") == F.col("reporter_bloc_iso3"))
        & (F.expr("make_date(year, 12, 31)") >= F.col("reporter_bloc_valid_from"))
        & (F.col("reporter_bloc_valid_to").isNull() | (F.expr("make_date(year, 12, 31)") <= F.col("reporter_bloc_valid_to"))),
        "left",
    )
    .withColumnRenamed("partner_iso3", "counterpart_iso3")
    .withColumn("year_end_date", F.expr("make_date(year, 12, 31)"))
    .withColumn("counterpart_is_project_country", F.col("counterpart_country_key").isNotNull())
    .withColumn(
        "counterpart_name",
        F.when(F.col("counterpart_project_country_name").isNotNull(), F.col("counterpart_project_country_name"))
        .otherwise(F.col("counterpart_iso3")),
    )
    # Comtrade rows are all bilateral (no world-aggregate); counterpart bloc resolved best-effort
    .withColumn("counterpart_group_code", F.lit("REST_OF_WORLD"))
    .withColumn("counterpart_group_name", F.lit("Rest of world"))
)

# Compute partner-share metrics for Comtrade rows
comtrade_window = Window.partitionBy("reporter_iso3", "year")
comtrade_enriched = (
    comtrade_enriched
    .withColumn(
        "reporter_year_exports_partner_sum_usd",
        F.sum(F.coalesce(F.col("exports_fob_usd"), F.lit(0.0))).over(comtrade_window),
    )
    .withColumn(
        "reporter_year_imports_partner_sum_usd",
        F.sum(F.coalesce(F.col("imports_cif_usd"), F.lit(0.0))).over(comtrade_window),
    )
    .withColumn(
        "reporter_year_total_trade_partner_sum_usd",
        F.sum(F.coalesce(F.col("total_trade_usd"), F.lit(0.0))).over(comtrade_window),
    )
    .withColumn(
        "export_partner_share_pct",
        F.when(
            F.col("reporter_year_exports_partner_sum_usd") > 0,
            F.col("exports_fob_usd") / F.col("reporter_year_exports_partner_sum_usd") * 100.0,
        ),
    )
    .withColumn(
        "import_partner_share_pct",
        F.when(
            F.col("reporter_year_imports_partner_sum_usd") > 0,
            F.col("imports_cif_usd") / F.col("reporter_year_imports_partner_sum_usd") * 100.0,
        ),
    )
    .withColumn(
        "total_trade_partner_share_pct",
        F.when(
            F.col("reporter_year_total_trade_partner_sum_usd") > 0,
            F.col("total_trade_usd") / F.col("reporter_year_total_trade_partner_sum_usd") * 100.0,
        ),
    )
    .withColumn("trade_source", F.lit("un_comtrade"))
    .withColumn("selected_source", F.lit("UN_COMTRADE_DIRECT"))
    .withColumn("is_comtrade_backed", F.lit(True))
    .withColumn("is_fallback", F.lit(False))
    .withColumn("coverage_method", F.lit("direct_hs6_aggregate"))
    .withColumn("loaded_at", F.lit(loaded_at))
)

# ─── Step D: Remove DOTS rows for cells where Comtrade is valid; UNION ────────
comtrade_cells = (
    comtrade_enriched
    .select("reporter_iso3", "year")
    .distinct()
    .withColumn("_comtrade_covered", F.lit(True))
)

dots_filtered = (
    dots_base
    .join(comtrade_cells, ["reporter_iso3", "year"], "left")
    .where(F.col("_comtrade_covered").isNull())
    .drop("_comtrade_covered")
)

# Shared column list — must be identical in both branches before UNION
shared_cols = [
    "reporter_country_key", "reporter_iso3", "reporter_name", "reporter_region",
    "reporter_bloc_code", "reporter_bloc_name",
    "counterpart_country_key", "counterpart_iso3", "counterpart_name",
    "counterpart_is_project_country", "counterpart_group_code", "counterpart_group_name",
    "year", "year_end_date",
    "exports_fob_usd", "imports_cif_usd", "total_trade_usd", "trade_balance_usd",
    "exports_billions_usd", "imports_billions_usd", "total_trade_billions_usd", "trade_balance_billions_usd",
    "reporter_year_exports_partner_sum_usd", "reporter_year_imports_partner_sum_usd",
    "reporter_year_total_trade_partner_sum_usd",
    "export_partner_share_pct", "import_partner_share_pct", "total_trade_partner_share_pct",
    "is_world_aggregate",
    "trade_source", "selected_source", "is_comtrade_backed", "is_fallback", "coverage_method",
    "hs6_code_count", "loaded_at",
]

fact_df = dots_filtered.select(*shared_cols).unionByName(comtrade_enriched.select(*shared_cols))

dots_rows = dots_filtered.count()
comtrade_rows = comtrade_enriched.count()
print(f"DOTS rows (non-Comtrade cells)  : {dots_rows:,}")
print(f"Comtrade rows (quality_flag=good): {comtrade_rows:,}")
print(f"Total fact rows                  : {dots_rows + comtrade_rows:,}")

fact_df.groupBy("trade_source", "selected_source").count().show()
fact_df.show(10, truncate=False)


In [ ]:
# Cell 6 - Write silver.fact_trade_partner_annual
spark.sql("DROP TABLE IF EXISTS silver.fact_trade_partner_annual")

(
    fact_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("reporter_iso3", "year")
    .saveAsTable("silver.fact_trade_partner_annual")
)

print("Write complete: silver.fact_trade_partner_annual")

# Source breakdown in the written table
spark.sql("""
    SELECT
        trade_source,
        COUNT(*) AS rows,
        COUNT(DISTINCT reporter_iso3) AS reporters,
        MIN(year) AS first_year,
        MAX(year) AS last_year
    FROM silver.fact_trade_partner_annual
    GROUP BY trade_source
    ORDER BY trade_source
""").show(truncate=False)

In [ ]:
# Cell 7 - Verification and sanity checks
print("Overall coverage:")
spark.sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT reporter_iso3) AS reporters,
        COUNT(DISTINCT counterpart_iso3) AS counterparts,
        MIN(year) AS earliest_year,
        MAX(year) AS latest_year,
        SUM(CASE WHEN is_world_aggregate THEN 1 ELSE 0 END) AS world_aggregate_rows,
        SUM(CASE WHEN counterpart_is_project_country THEN 1 ELSE 0 END) AS project_partner_rows,
        ROUND(SUM(exports_fob_usd) / 1e12, 2) AS exports_trillions_usd,
        ROUND(SUM(imports_cif_usd) / 1e12, 2) AS imports_trillions_usd
    FROM silver.fact_trade_partner_annual
""").show()

print("Reporter coverage:")
spark.sql("""
    SELECT
        reporter_iso3,
        reporter_name,
        COUNT(*) AS rows,
        COUNT(DISTINCT counterpart_iso3) AS counterparts,
        COUNT(DISTINCT year) AS years,
        ROUND(SUM(CASE WHEN NOT is_world_aggregate THEN total_trade_usd ELSE 0 END) / 1e9, 2) AS partner_trade_billions_usd
    FROM silver.fact_trade_partner_annual
    GROUP BY reporter_iso3, reporter_name
    ORDER BY partner_trade_billions_usd DESC
""").show(25, truncate=False)

print("Cameroon top partners in latest year, excluding world aggregate:")
spark.sql("""
    SELECT
        counterpart_iso3,
        counterpart_name,
        ROUND(exports_billions_usd, 2) AS exports_billions_usd,
        ROUND(imports_billions_usd, 2) AS imports_billions_usd,
        ROUND(total_trade_billions_usd, 2) AS total_trade_billions_usd,
        ROUND(total_trade_partner_share_pct, 1) AS total_trade_share_pct
    FROM silver.fact_trade_partner_annual
    WHERE reporter_iso3 = 'CMR'
      AND year = (SELECT MAX(year) FROM silver.fact_trade_partner_annual)
      AND NOT is_world_aggregate
    ORDER BY total_trade_usd DESC
    LIMIT 15
""").show(truncate=False)

print("Latest year bloc-to-bloc trade, project partners only:")
spark.sql("""
    SELECT
        reporter_bloc_code,
        counterpart_group_code,
        COUNT(DISTINCT reporter_iso3) AS reporters,
        COUNT(DISTINCT counterpart_iso3) AS counterpart_countries,
        ROUND(SUM(exports_fob_usd) / 1e9, 2) AS exports_billions_usd,
        ROUND(SUM(imports_cif_usd) / 1e9, 2) AS imports_billions_usd,
        ROUND(SUM(total_trade_usd) / 1e9, 2) AS total_trade_billions_usd
    FROM silver.fact_trade_partner_annual
    WHERE year = (SELECT MAX(year) FROM silver.fact_trade_partner_annual)
      AND counterpart_is_project_country
    GROUP BY reporter_bloc_code, counterpart_group_code
    ORDER BY reporter_bloc_code, counterpart_group_code
""").show(truncate=False)

In [ ]:
# Cell 7b - Source-seam audit: show which source covers each reporter×year
# Useful for QA and for the coverage audit notebook (13).
spark.sql("""
    SELECT
        reporter_iso3,
        year,
        trade_source,
        COUNT(DISTINCT counterpart_iso3) AS partners,
        ROUND(SUM(CASE WHEN NOT is_world_aggregate THEN total_trade_usd ELSE 0 END) / 1e9, 3) AS trade_billions_usd
    FROM silver.fact_trade_partner_annual
    WHERE reporter_iso3 IN (
        SELECT DISTINCT reporter_iso3
        FROM silver.fact_trade_partner_annual
        WHERE trade_source = 'un_comtrade'
    )
    GROUP BY reporter_iso3, year, trade_source
    ORDER BY reporter_iso3, year
""").show(200, truncate=False)

In [ ]:
# Cell 8 - Partner concentration helper view
# HHI is calculated on individual partners only. The W00 world aggregate row is excluded.
partner_hhi_df = (
    spark.table("silver.fact_trade_partner_annual")
    .where(~F.col("is_world_aggregate"))
    .groupBy("reporter_iso3", "reporter_name", "reporter_bloc_code", "year")
    .agg(
        F.sum(F.pow(F.coalesce(F.col("export_partner_share_pct"), F.lit(0.0)) / 100.0, 2)).alias("export_partner_hhi"),
        F.sum(F.pow(F.coalesce(F.col("import_partner_share_pct"), F.lit(0.0)) / 100.0, 2)).alias("import_partner_hhi"),
        F.sum(F.pow(F.coalesce(F.col("total_trade_partner_share_pct"), F.lit(0.0)) / 100.0, 2)).alias("total_trade_partner_hhi"),
        F.countDistinct("counterpart_iso3").alias("partner_count"),
    )
)

print("Most concentrated latest-year partner structures:")
partner_hhi_df.where(F.col("year") == END_YEAR).orderBy(F.desc("total_trade_partner_hhi")).show(25, truncate=False)

spark.sql("DROP TABLE IF EXISTS silver.trade_partner_concentration_annual")
(partner_hhi_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.trade_partner_concentration_annual"))

print("Write complete: silver.trade_partner_concentration_annual")